# Vector-Quantized Variational Autoencoder 2
# VQ-VAE 2

Source:<P>
https://github.com/rosinality/vq-vae-2-pytorch

Adapted:<P>
    Antonio Esteves @ UMinho, May 2024

## Import necessary libraries

In [ ]:
import torch
from   torch              import nn, optim
from   torch.utils.data   import DataLoader, Dataset
from   torchvision        import datasets, transforms, utils
from   torchvision.utils  import save_image
from   torch.nn           import functional    as F
from   torch.optim        import lr_scheduler

from   dataset            import ImageFileDataset, CodeRow
from   collections        import namedtuple
from   tqdm.notebook      import trange, tqdm
from   math               import sqrt, cos, pi, floor, sin
from   functools          import partial, lru_cache
from   typing             import Tuple, Dict, List
import numpy              as     np
from   pathlib            import Path
from   natsort            import natsorted
from   PIL                import Image
import matplotlib.pyplot  as     plt
import pickle
import lmdb
import os
import wandb
import yaml
import random
import time

try:
    from apex import amp
except ImportError:
    amp = None


## Import W&B and Login

In [ ]:
wandb.login()

## Read the configuration file

In [ ]:
CONFIG_FILE                 = 'config/config1.yaml'
LOAD_VQVAE_MODEL            = False
SKIP_TRAIN_VQVAE_MODEL      = False
LOAD_PIXELSNAIL_MODEL       = False
SKIP_TRAIN_PIXELSNAIL_MODEL = False
GENERATE_NEW_IMAGES         = False
calculate_statistics        = False
sample_size                 = 8

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

BASE_FILE_NAME                        = config["exp_params"]["vqvae_save_file"]
config["exp_params"]["eps"]           = float(config["exp_params"]["eps"])
config["exp_params"]["learning_rate"] = float(config["exp_params"]["learning_rate"])
config["exp_params"]["pixelsnail_lr"] = float(config["exp_params"]["pixelsnail_lr"])

In [ ]:
print('model_params:')
model_params = config['model_params']
for key, value in model_params.items():
    print(f'\t{key}: {value}')

print('exp_params:')
exp_params = config['exp_params']
for key, value in exp_params.items():
    print(f'\t{key}: {value}')

## Track metadata and hyperparameters with `wandb.init`

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = dict(
    model                       = config["model_params"]["name"],
    config                      = CONFIG_FILE,
    VQVAE_file                  = f'{config["exp_params"]["vqvae_save_file"]}.pth',
    PixelSNAIL_file             = f'{config["exp_params"]["pixelsnail_save_file"]}.pth',
    hidden_channels             = config["model_params"]["hidden_channels"],
    res_hidden_channels         = config["model_params"]["res_hidden_channels"],
    num_residual_blocks         = config["model_params"]["num_residual_blocks"],
    stride_top                  = config["model_params"]["stride_top"],
    stride_bottom               = config["model_params"]["stride_bottom"],
    embedding_dim               = config["model_params"]["embedding_dim"],
    embedding_number            = config["model_params"]["embedding_number"],
    patch_size                  = config["model_params"]["patch_size"],
    num_channels                = config["model_params"]["num_channels"],
    pixelsnail_channels         = config["model_params"]["pixelsnail_channels"],
    pixelsnail_res_blocks       = config["model_params"]["pixelsnail_res_blocks"],
    pixelsnail_res_channels     = config["model_params"]["pixelsnail_res_channels"],
    pixelsnail_out_res_blocks   = config["model_params"]["pixelsnail_out_res_blocks"],
    pixelsnail_cond_res_blocks  = config["model_params"]["pixelsnail_cond_res_blocks"],
    pixelsnail_amp_opt_level    = config["model_params"]["pixelsnail_amp_opt_level"],
    dataset                     = config["exp_params"]["dataset"],
    train_batch_size            = config["exp_params"]["train_batch_size"],
    val_batch_size              = config["exp_params"]["val_batch_size"],
    test_batch_size             = config["exp_params"]["test_batch_size"],
    epochs                      = config["exp_params"]["epochs"],
    optimizer                   = config["exp_params"]["optimizer"],
    scheduler                   = config["exp_params"]["vqvae_scheduler"],
    learning_rate               = config["exp_params"]["learning_rate"],
    decay                       = config["exp_params"]["decay"],
    eps                         = config["exp_params"]["eps"],
    beta                        = config["exp_params"]["beta"],
    pixelsnail_optimizer        = config["exp_params"]["pixelsnail_optimizer"],
    pixelsnail_scheduler        = config["exp_params"]["pixelsnail_scheduler"],
    pixelsnail_lr               = config["exp_params"]["pixelsnail_lr"],
    pixelsnail_epochs           = config["exp_params"]["pixelsnail_epochs"],
    pixelsnail_train_batch_size = config["exp_params"]["pixelsnail_train_batch_size"],
    pixelsnail_dropout          = config["exp_params"]["pixelsnail_dropout"],
)

In [ ]:
print('W&B logged parameters:')
for key, value in config_wandb.items():
    print(f'\t{key}: {value}')

In [ ]:
wandb.init(project='VQVAE2_CelebA', entity='ajesteves', config=config_wandb)

## Initializations

In [ ]:
print(torch.__version__)

# setup device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

# Setup path to celebA dataset folder

data_path = Path(config["exp_params"]["data_path"])

train_dir = data_path / "train"
val_dir   = data_path / "val"
test_dir  = data_path / "test"

## Exploring the CelebA dataset

In [ ]:
# Get all training image paths
train_image_list = list(train_dir.glob("*.jpg"))

print(f'Training set size: {len(train_image_list)}')

# Get all validation image paths
val_image_list = list(val_dir.glob("*.jpg"))

print(f'Validation set size: {len(val_image_list)}')

# Get all test image paths
test_image_list = list(test_dir.glob("*.jpg"))

print(f'Test set size: {len(test_image_list)}')

In [ ]:
# Visualizing random images

# set seed

random.seed(42)

# Pick a random image path

rand_image_path = random.choice(train_image_list)
print(f'Randomly selected image:\n\t{rand_image_path}')

# Open the image using pillow library

img = Image.open(rand_image_path)

# Show the image and print image metadata

print(f'Random image path:   {rand_image_path}')
print(f'Random image height: {img.height}')
print(f'Random image width:  {img.width}')

display(img)

In [ ]:
# Visualize an image using matplotlib

# convert the image 'img' to a numpy array

img_array = np.asarray(img)

# plot the image with matplotlib

plt.figure(figsize=(10,7))
plt.imshow(img_array)
plt.title(f'Image shape: {img_array.shape} (height,width,channels)')
plt.axis(False)

In [ ]:
train_mean     = torch.tensor([0.5184, 0.4153, 0.3617])
train_variance = torch.tensor([0.0890, 0.0715, 0.0682])
train_stddev   = torch.tensor([0.2983, 0.2674, 0.2611])

print(f'Training data mean:     {train_mean}')
print(f'Training data variance: {train_variance}')
print(f'Training data stddev:   {train_stddev}')

## Transforming the Data

Before we can use our data with PyTorch:
1. Convert the images to tensors using `torchvision.transforms`([documentation](https://pytorch.org/vision/0.16//transforms.html))
2. Create a custom Dataset from the images in a folder and associated attributes using `torch.utils.data.Dataset`
3. Create training, validation and test datasets
4. Convert the `Datasets` into `torch.utils.data.DataLoaders`

### (i) Convert images to tensors

In [ ]:
train_transforms = transforms.Compose(
    [
    #transforms.RandomHorizontalFlip(),
    transforms.Resize(config["model_params"]["patch_size"]),
    transforms.CenterCrop(config["model_params"]["patch_size"]),
    transforms.ToTensor(),
    #transforms.Normalize(train_mean, train_stddev),
    ]
)

val_transforms = transforms.Compose(
    [
    #transforms.RandomHorizontalFlip(),
    transforms.Resize(config["model_params"]["patch_size"]),
    transforms.CenterCrop(config["model_params"]["patch_size"]),
    transforms.ToTensor(),
    #transforms.Normalize(train_mean, train_stddev),
    ]
)

In [ ]:
img_tensor = train_transforms(img)

print(img_tensor.shape)
print(img_tensor)

In [ ]:
# Visualize transformed images

def plot_transformed_images(
    image_paths,
    transform,
    n           = 3,
    seed        = None ):
  """
  The functions selects 'n' random images from 'image_paths', load and transform them,
  then it plots the original images vs the transformed representation.
  """
  # set the random seed if it is not None
  if seed:
    random.seed(seed)

  # randomly select 'n' image paths
  rand_image_paths = random.sample(image_paths,k=n)

  # iterate throuth the 'n' selected paths
  for image_path in rand_image_paths:

    with Image.open(image_path) as f:

      # original image
      fig,ax = plt.subplots(nrows=1,ncols=2)
      ax[0].imshow(f)
      ax[0].set_title(f'Original\nSize: {f.size}')
      ax[0].axis(False)

      # transform the original image and plot the transformed representation
      transformed_image = transform(f).permute(1, 2, 0) # we need to convert C,H,W (pytorch) to H,W,C (plt)
      ax[1].imshow(transformed_image)
      ax[1].set_title(f'Transformed\nShape: {transformed_image.shape}')
      ax[1].axis(False)

      fig.suptitle(f'Class: {image_path.parent.stem}',fontsize=16)

In [ ]:
plot_transformed_images(
    image_paths = train_image_list,
    transform   = train_transforms,
    n           = 3
)

### (ii) Create a custom Dataset from the images in a folder and the associated attributes using `torch.utils.data.Dataset`

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc = os.path.join(self.root_dir, self.total_images[idx])
        image   = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

In [ ]:
# Create training and validation datasets

train_data = CustomDataSet(
    train_dir,
    transform = train_transforms
)

val_data = CustomDataSet(
    val_dir,
    transform = val_transforms
)

test_data = CustomDataSet(
    test_dir,
    transform = val_transforms
)

train_data_len = len(train_data)
val_data_len   = len(val_data)
test_data_len  = len(test_data)

print(train_data_len)
print(val_data_len)
print(test_data_len)

In [ ]:
# Visualize some samples from the created training set

# Indexing the 'train_data' Dataset to get a single image

img  = train_data[1]

print(f'Image tensor:\n{img}')
print(f'Image shape: {img.shape}')
print(f'Image data type: {img.dtype}')

In [ ]:
# rearrange the order of dimensions on image tensor for using matplot lib
img_permuted = img.permute(1, 2, 0) # C,H,W --> H,W,C

# print shapes
print(f'Original image shape (pytorch tensor):   {img.shape}')
print(f'Permute image shape (matplotlib format): {img_permuted.shape}')

# plot the image

plt.figure(figsize=(10,7))
plt.imshow(img_permuted)
plt.axis(False)
plt.title(f'An image from training set')

### (iii) Convert the `Dataset` into a `torch.utils.data.DataLoader`

A DataLoader helps us to iterate trough the dataset em get a batch of samples each time.

Relevant configuration parameters:

```python
config["exp_params"]["train_batch_size"]
config["exp_params"]["val_batch_size"]
config["exp_params"]["test_batch_size"]
config["exp_params"]["num_workers"]
config["exp_params"]["pin_memory"]
```

In [ ]:
# Convert each 'Dataset' into a 'DataLoader'

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset     = train_data,
    batch_size  = config["exp_params"]["train_batch_size"],
    shuffle     = True,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

val_dataloader = DataLoader(
    dataset     = val_data,
    batch_size  = config["exp_params"]["val_batch_size"],
    shuffle     = False,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

test_dataloader = DataLoader(
    dataset     = test_data,
    batch_size  = config["exp_params"]["test_batch_size"],
    shuffle     = False,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

In [ ]:
print(train_dataloader)
print(val_dataloader)
print(test_dataloader)

print(f'Number of batches in a training epoch:   {len(train_dataloader)}')
print(f'Number of batches in a validation epoch: {len(val_dataloader)}')
print(f'Number of batches in a testing epoch:    {len(test_dataloader)}')

In [ ]:
# Print metadata about a sample retrieved from the training DataLoader

imgs = next(iter(train_dataloader))

print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

In [ ]:
if calculate_statistics == True:
    train_mean = 0.0
    for images in train_dataloader:
        batch_samples = images.size(0)
        images        = images.view(batch_samples, images.size(1), -1)
        train_mean   += images.mean(2).sum(0)
        print('.',end='')
    train_mean = train_mean / len(train_dataloader.dataset)

    train_variance   = 0.0
    pixel_count      = 0
    for images in train_dataloader:
        batch_samples   = images.size(0)
        images          = images.view(batch_samples, images.size(1), -1)
        train_variance += ((images - train_mean.unsqueeze(1))**2).sum([0,2])
        pixel_count    += images.nelement() / images.size(1)
        print('*',end='')
    train_variance /= pixel_count
    train_stddev = torch.sqrt(train_variance)

else:
    train_mean     = torch.tensor([0.5184, 0.4153, 0.3617])
    train_variance = torch.tensor([0.0890, 0.0715, 0.0682])
    train_stddev   = torch.tensor([0.2983, 0.2674, 0.2611])

print(f'Training data mean:     {train_mean}')
print(f'Training data variance: {train_variance}')
print(f'Training data stddev:   {train_stddev}')

## Learning rate schedulers

In [ ]:
class CosineLR(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, step_size):
        self.lr_min    = lr_min
        self.lr_max    = lr_max
        self.step_size = step_size
        self.iteration = 0

        super().__init__(optimizer, -1)

    def get_lr(self):
        lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (
            1 + cos(self.iteration / self.step_size * pi)
        )
        self.iteration += 1

        if self.iteration == self.step_size:
            self.iteration = 0

        return [lr for base_lr in self.base_lrs]

In [ ]:
class PowerLR(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, warmup):
        self.lr_min    = lr_min
        self.lr_max    = lr_max
        self.warmup    = warmup
        self.iteration = 0

        super().__init__(optimizer, -1)

    def get_lr(self):
        if self.iteration < self.warmup:
            lr = (
                self.lr_min + (self.lr_max - self.lr_min) / self.warmup * self.iteration
            )

        else:
            lr = self.lr_max * (self.iteration - self.warmup + 1) ** -0.5

        self.iteration += 1

        return [lr for base_lr in self.base_lrs]

In [ ]:
class SineLR(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, step_size):
        self.lr_min    = lr_min
        self.lr_max    = lr_max
        self.step_size = step_size
        self.iteration = 0

        super().__init__(optimizer, -1)

    def get_lr(self):
        lr = self.lr_min + (self.lr_max - self.lr_min) * sin(
            self.iteration / self.step_size * pi
        )
        self.iteration += 1

        if self.iteration == self.step_size:
            self.iteration = 0

        return [lr for base_lr in self.base_lrs]

In [ ]:
class LinearLR(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, warmup, step_size):
        self.lr_min    = lr_min
        self.lr_max    = lr_max
        self.step_size = step_size
        self.warmup    = warmup
        self.iteration = 0

        super().__init__(optimizer, -1)

    def get_lr(self):
        if self.iteration < self.warmup:
            lr = self.lr_max

        else:
            lr = self.lr_max + (self.iteration - self.warmup) * (
                self.lr_min - self.lr_max
            ) / (self.step_size - self.warmup)
        self.iteration += 1

        if self.iteration == self.step_size:
            self.iteration = 0

        return [lr for base_lr in self.base_lrs]

In [ ]:
class CLR(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, step_size):
        self.epoch      = 0
        self.lr_min     = lr_min
        self.lr_max     = lr_max
        self.current_lr = lr_min
        self.step_size  = step_size

        super().__init__(optimizer, -1)

    def get_lr(self):
        cycle = floor(1 + self.epoch / (2 * self.step_size))
        x     = abs(self.epoch / self.step_size - 2 * cycle + 1)
        lr    = self.lr_min + (self.lr_max - self.lr_min) * max(0, 1 - x)
        self.current_lr  = lr
        self.epoch      += 1

        return [lr for base_lr in self.base_lrs]

In [ ]:
class Warmup(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, model_dim, factor=1, warmup=16000):
        self.optimizer = optimizer
        self.model_dim = model_dim
        self.factor    = factor
        self.warmup    = warmup
        self.iteration = 0

        super().__init__(optimizer, -1)

    def get_lr(self):
        self.iteration += 1
        lr = (
            self.factor
            * self.model_dim ** (-0.5)
            * min(self.iteration ** (-0.5), self.iteration * self.warmup ** (-1.5))
        )

        return [lr for base_lr in self.base_lrs]

In [ ]:
# Copyright 2019 fastai
#
# Code borrowed from https://github.com/fastai/fastai 
# and changed to make it run like PyTorch LR scheduler

def anneal_linear(start, end, proportion):
    return start + proportion * (end - start)


def anneal_cos(start, end, proportion):
    cos_val = cos(pi * proportion) + 1

    return end + (start - end) / 2 * cos_val

In [ ]:
# Copyright 2019 fastai
#
# Code borrowed from https://github.com/fastai/fastai 
# and changed to make it run like PyTorch LR scheduler

class Phase:

    def __init__(self, start, end, n_iter, anneal_fn):
        self.start, self.end = start, end
        self.n_iter          = n_iter
        self.anneal_fn       = anneal_fn
        self.n               = 0

    def step(self):
        self.n += 1

        return self.anneal_fn(self.start, self.end, self.n / self.n_iter)

    def reset(self):
        self.n = 0

    @property
    def is_done(self):
        return self.n >= self.n_iter

In [ ]:
# Copyright 2019 fastai
#
# Code borrowed from https://github.com/fastai/fastai 
# and changed to make it run like PyTorch LR scheduler

class CycleAnnealScheduler:

    def __init__(
        self, optimizer, lr_max, lr_divider, cut_point, step_size, momentum=None
        ):
        self.lr_max     = lr_max
        self.lr_divider = lr_divider
        self.cut_point  = step_size // cut_point
        self.step_size  = step_size
        self.iteration  = 0
        self.cycle_step = int(step_size * (1 - cut_point / 100) / 2)
        self.momentum   = momentum
        self.optimizer  = optimizer

    def get_lr(self):
        if self.iteration > 2 * self.cycle_step:
            cut = (self.iteration - 2 * self.cycle_step) / (
                self.step_size - 2 * self.cycle_step
            )
            lr = self.lr_max * (1 + (cut * (1 - 100) / 100)) / self.lr_divider

        elif self.iteration > self.cycle_step:
            cut = 1 - (self.iteration - self.cycle_step) / self.cycle_step
            lr = self.lr_max * (1 + cut * (self.lr_divider - 1)) / self.lr_divider

        else:
            cut = self.iteration / self.cycle_step
            lr = self.lr_max * (1 + cut * (self.lr_divider - 1)) / self.lr_divider

        return lr

    def get_momentum(self):
        if self.iteration > 2 * self.cycle_step:
            momentum = self.momentum[0]

        elif self.iteration > self.cycle_step:
            cut = 1 - (self.iteration - self.cycle_step) / self.cycle_step
            momentum = self.momentum[0] + cut * (self.momentum[1] - self.momentum[0])

        else:
            cut = self.iteration / self.cycle_step
            momentum = self.momentum[0] + cut * (self.momentum[1] - self.momentum[0])

        return momentum

    def step(self):
        lr = self.get_lr()

        if self.momentum is not None:
            momentum = self.get_momentum()

        self.iteration += 1

        if self.iteration == self.step_size:
            self.iteration = 0

        for group in self.optimizer.param_groups:
            group['lr'] = lr

            if self.momentum is not None:
                group['betas'] = (momentum, group['betas'][1])

        return lr

In [ ]:
# Copyright 2019 fastai
#
# Code borrowed from https://github.com/fastai/fastai 
# and changed to make it run like PyTorch LR scheduler

class CycleScheduler:

    def __init__(
        self,
        optimizer,
        lr_max,
        n_iter,
        momentum          = (0.95, 0.85),
        divider           = 25,
        warmup_proportion = 0.3,
        phase             = ('linear', 'cos'),
        ):
        self.optimizer = optimizer

        phase1 = int(n_iter * warmup_proportion)
        phase2 = n_iter - phase1
        lr_min = lr_max / divider

        phase_map = {'linear': anneal_linear, 'cos': anneal_cos}

        self.lr_phase = [
            Phase(lr_min, lr_max, phase1, phase_map[phase[0]]),
            Phase(lr_max, lr_min / 1e4, phase2, phase_map[phase[1]]),
        ]

        self.momentum = momentum

        if momentum is not None:
            mom1, mom2 = momentum
            self.momentum_phase = [
                Phase(mom1, mom2, phase1, phase_map[phase[0]]),
                Phase(mom2, mom1, phase2, phase_map[phase[1]]),
            ]

        else:
            self.momentum_phase = []

        self.phase = 0

    def step(self):
        lr = self.lr_phase[self.phase].step()

        if self.momentum is not None:
            momentum = self.momentum_phase[self.phase].step()

        else:
            momentum = None

        for group in self.optimizer.param_groups:
            group['lr'] = lr

            if self.momentum is not None:
                if 'betas' in group:
                    group['betas'] = (momentum, group['betas'][1])

                else:
                    group['momentum'] = momentum

        if self.lr_phase[self.phase].is_done:
            self.phase += 1

        if self.phase >= len(self.lr_phase):
            for phase in self.lr_phase:
                phase.reset()

            for phase in self.momentum_phase:
                phase.reset()

            self.phase = 0

        return lr, momentum

In [ ]:
# Copyright 2019 fastai
#
# Code borrowed from https://github.com/fastai/fastai 
# and changed to make it run like PyTorch LR scheduler

class LRFinder(lr_scheduler._LRScheduler):

    def __init__(self, optimizer, lr_min, lr_max, step_size, linear=False):
        ratio          = lr_max / lr_min
        self.linear    = linear
        self.lr_min    = lr_min
        self.lr_mult   = (ratio / step_size) if linear else ratio ** (1 / step_size)
        self.iteration = 0
        self.lrs       = []
        self.losses    = []

        super().__init__(optimizer, -1)

    def get_lr(self):
        lr = (
            self.lr_mult * self.iteration
            if self.linear
            else self.lr_mult ** self.iteration
        )
        lr = self.lr_min + lr if self.linear else self.lr_min * lr

        self.iteration += 1
        self.lrs.append(lr)

        return [lr for base_lr in self.base_lrs]

    def record(self, loss):
        self.losses.append(loss)

    def save(self, filename):
        with open(filename, 'w') as f:
            for lr, loss in zip(self.lrs, self.losses):
                f.write('{},{}\n'.format(lr, loss))

### Quantizer module

In [ ]:
class Quantize(nn.Module):

    def __init__(self, embedding_dim, embedding_number, decay=0.99, eps=1e-5):
        super().__init__()

        self.embedding_dim    = embedding_dim
        self.embedding_number = embedding_number
        self.decay            = decay
        self.eps              = eps

        embed = torch.randn(embedding_dim, embedding_number)
        self.register_buffer("embed", embed)
        self.register_buffer("cluster_size", torch.zeros(embedding_number))
        self.register_buffer("embed_avg", embed.clone())

    def forward(self, input): 
        # input = [BS, LATENT_H, LATENT_W, EMBED_DIM]

        flatten = input.reshape(-1, self.embedding_dim)
        # flatten = [BS * LATENT_H * LATENT_W, EMBED_DIM]

        dist = (
            flatten.pow(2).sum(1, keepdim=True)
            - 2 * flatten @ self.embed
            + self.embed.pow(2).sum(0, keepdim=True)
        )
        _, embed_ind = (-dist).max(1)
        # embed_ind = [BS * LATENT_H * LATENT_W]

        embed_onehot = F.one_hot(embed_ind, self.embedding_number).type(flatten.dtype)
        embed_ind    = embed_ind.view(*input.shape[:-1])
        # embed_ind = [BS, LATENT_H, LATENT_W]

        quantize     = self.embed_code(embed_ind)

        if self.training:
            embed_onehot_sum = embed_onehot.sum(0)
            embed_sum        = flatten.transpose(0, 1) @ embed_onehot

            self.cluster_size.data.mul_(self.decay).add_(
                embed_onehot_sum, alpha=1 - self.decay
            )
            self.embed_avg.data.mul_(self.decay).add_(embed_sum, alpha=1 - self.decay)
            n = self.cluster_size.sum()
            cluster_size = (
                (self.cluster_size + self.eps) / (n + self.embedding_number * self.eps) * n
            )
            embed_normalized = self.embed_avg / cluster_size.unsqueeze(0)
            self.embed.data.copy_(embed_normalized)

        diff     = (quantize.detach() - input).pow(2).mean()
        quantize = input + (quantize - input).detach()

        # embed_ind = [BS, LATENT_H, LATENT_W]
        return quantize, diff, embed_ind

    def embed_code(self, embed_id):
        return F.embedding(embed_id, self.embed.transpose(0, 1))

### Residual Block

In [ ]:
class ResBlock(nn.Module):

    def __init__(self, in_channels, channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.ReLU(),
            nn.Conv2d(in_channels, channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, in_channels, 1),
        )

    def forward(self, input):
        out = self.conv(input)
        out += input
        return out

### Encoder module

In [ ]:
class Encoder(nn.Module):

    def __init__(self, in_channels, channels, n_res_blocks, n_res_channels, stride):
        super().__init__()

        if stride == 4:
            blocks = [
                nn.Conv2d(in_channels, channels // 2, 4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels // 2, channels, 4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels, channels, 3, padding=1),
            ]

        elif stride == 2:
            blocks = [
                nn.Conv2d(in_channels, channels // 2, 4, stride=2, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(channels // 2, channels, 3, padding=1),
            ]

        for i in range(n_res_blocks):
            blocks.append(ResBlock(channels, n_res_channels))

        blocks.append(nn.ReLU(inplace=True))

        self.blocks = nn.Sequential(*blocks)

    def forward(self, input):
        return self.blocks(input)

### Decoder module

In [ ]:
class Decoder(nn.Module):

    def __init__(
        self, in_channels, out_channels, channels, n_res_blocks, n_res_channels, stride
    ):
        super().__init__()

        blocks = [nn.Conv2d(in_channels, channels, 3, padding=1)]

        for i in range(n_res_blocks):
            blocks.append(ResBlock(channels, n_res_channels))

        blocks.append(nn.ReLU(inplace=True))

        if stride == 4:
            blocks.extend(
                [
                    nn.ConvTranspose2d(channels, channels // 2, 4, stride=2, padding=1),
                    nn.ReLU(inplace=True),
                    nn.ConvTranspose2d(
                        channels // 2, out_channels, 4, stride=2, padding=1
                    ),
                ]
            )

        elif stride == 2:
            blocks.append(
                nn.ConvTranspose2d(channels, out_channels, 4, stride=2, padding=1)
            )

        self.blocks = nn.Sequential(*blocks)

    def forward(self, input):
        return self.blocks(input)

### VQ-VAE 2 main class

In [ ]:
class VQVAE2(nn.Module):
    def __init__(
        self,
        in_channels      = 3,
        channels         = 128,
        n_res_blocks     = 2,
        n_res_channels   = 32,
        embedding_dim    = 64,
        embedding_number = 512,
        decay            = 0.99,
        eps              = 1e-5,
        stride_top       = 2,
        stride_bottom    = 4,
        ):
        super().__init__()

        self.enc_b = Encoder(
            in_channels,
            channels, 
            n_res_blocks, 
            n_res_channels, 
            stride=stride_bottom
        )
        self.enc_t = Encoder(
            channels, 
            channels,
            n_res_blocks,
            n_res_channels,
            stride=stride_top
        )
        self.quantize_conv_t = nn.Conv2d(channels, embedding_dim, 1)
        self.quantize_t      = Quantize(embedding_dim, embedding_number, decay=decay, eps=eps)
        self.dec_t           = Decoder(
            embedding_dim,
            embedding_dim,
            channels,
            n_res_blocks,
            n_res_channels,
            stride=stride_top
        )
        self.quantize_conv_b = nn.Conv2d(embedding_dim + channels, embedding_dim, 1)
        self.quantize_b      = Quantize(embedding_dim, embedding_number, decay=decay, eps=eps)
        self.upsample_t      = nn.ConvTranspose2d(
            embedding_dim, embedding_dim, 4, stride=2, padding=1
        )
        self.dec = Decoder(
            embedding_dim + embedding_dim,
            in_channels,
            channels,
            n_res_blocks,
            n_res_channels,
            stride = stride_bottom,
        )

    def forward(self, input):
        quant_t, quant_b, diff, id_t, id_b = self.encode(input)
        dec     = self.decode(quant_t, quant_b)
        return dec, diff, id_t, id_b

    def encode(self, input):
        enc_b   = self.enc_b(input)
        enc_t   = self.enc_t(enc_b)

        quant_t = self.quantize_conv_t(enc_t).permute(0, 2, 3, 1)
        quant_t, diff_t, id_t = self.quantize_t(quant_t)
        quant_t = quant_t.permute(0, 3, 1, 2)
        diff_t  = diff_t.unsqueeze(0)

        dec_t   = self.dec_t(quant_t)
        enc_b   = torch.cat([dec_t, enc_b], 1)

        quant_b = self.quantize_conv_b(enc_b).permute(0, 2, 3, 1)
        quant_b, diff_b, id_b = self.quantize_b(quant_b)
        quant_b = quant_b.permute(0, 3, 1, 2)
        diff_b  = diff_b.unsqueeze(0)

        return quant_t, quant_b, diff_t + diff_b, id_t, id_b

    def decode(self, quant_t, quant_b):
        upsample_t = self.upsample_t(quant_t)
        quant      = torch.cat([upsample_t, quant_b], 1)
        dec        = self.dec(quant)
        return dec

    def decode_code(self, code_t, code_b):
        quant_t = self.quantize_t.embed_code(code_t)
        quant_t = quant_t.permute(0, 3, 1, 2)
        quant_b = self.quantize_b.embed_code(code_b)
        quant_b = quant_b.permute(0, 3, 1, 2)
        dec     = self.decode(quant_t, quant_b)
        return dec

## Functions to save and load a model to/from file

In [ ]:
def save_model_and_results(model, results, hyperparameters, file_name):
    results_to_save = {
        'model':           model.state_dict(),
        'results':         results,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(model, file_name, config, device):
    '''
    Given an instance of a model, loads from file 'file_name':
    (i)   the model weights,
    (ii)  the results obtained during the model training and
    (iii) the training hyperparameters used to train the model,
    and put the model on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    model.load_state_dict(results_loaded['model'])
    model.to(device)

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']

## Functions to train the VQ-VAE 2

In [ ]:
def train_step_vqvae2(
        model:          torch.nn.Module,
        dataloader:     torch.utils.data.DataLoader,
        epoch:          int,
        log_interval:   int,
        optimizer:      torch.optim.Optimizer,
        scheduler:      torch.optim.lr_scheduler,
        results:        Dict[str, List[float]],
        train_variance: torch.Tensor,
        beta:           float        = 0.25,
        device:         torch.device = device,
    ):
    """
	Trains the VQ-VAE2 'model' for a single epoch.

	Turns the model into the training mode and then
	runs through all of the required training steps (forward
	pass, loss calculation, optimizer step).

	Arguments:
		model:          A VQ-VAE2 model to be trained.
		dataloader:     A DataLoader instance for the model to be trained on.
		epoch:          The current training epoch number.
        log_interval:   Interval (in batches) between successive logs.
		optimizer:      A PyTorch optimizer to help minimize the loss function.
        scheduler:      The learning rate scheduler to apply.
		results:        Dictionary to append the training results.
        train_variance: The training dataset variance.
        beta:           Scalar that defines the weight of the commitment term
                        of the loss (variable 'beta' in equation 3 of VQ-VAE paper).
		device:         A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of loss, reconstruction loss (and perplexity) metrics.
        In the form (loss, reconstruction_loss, perplexity).
    """

    train_loss         = 0
    train_reconst_loss = 0
    train_perplexity_t = 0
    train_perplexity_b = 0

    # Put the model in train mode
    model.train()

    for i, x in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):

        # 1. Send data to target device
        x = x.to(device)

        # 2. Reset the gradients of the loss
        model.zero_grad()

        # 3. Forward pass through the model
        x_pred, latent_loss, encode_indices_t, encode_indices_b = model(x)

        # 4. Calculate the loss
        reconst_loss = torch.mean((x_pred - x)**2) / train_variance
        loss         = reconst_loss + beta * latent_loss

        # 5. Calculate the gradient of the loss relative to all model parameters
        loss.backward()

        # 6. Update parameters using the calculated gradients
        optimizer.step()

        # 7. Accumulate the metrics
        train_loss         += loss.item()
        train_reconst_loss += reconst_loss.item()

        # 7a. Calculate the perplexity = Exp ( −Mean { Log [p(e) + epsilon] } )

        # encode_indices_t shape = [BS,TOP_LATENT_H,TOP_LATENT_W]
        size_encode_indices_t = encode_indices_t.shape[0]*encode_indices_t.shape[1]*encode_indices_t.shape[2]
        encode_indices_t_flat = encode_indices_t.view(size_encode_indices_t)
        # encode_indices_t shape = [BS * TOP_LATENT_H * TOP_LATENT_W]
        embed_mean_probs    = torch.bincount(encode_indices_t_flat)/size_encode_indices_t
        embed_entropy       = embed_mean_probs * torch.log(embed_mean_probs + 1e-10)
        perplexity_t        = torch.exp(-torch.sum(embed_entropy))
        train_perplexity_t += perplexity_t.item()

        # encode_indices_b shape = [BS,BOTTOM_LATENT_H,BOTTOM_LATENT_W]
        size_encode_indices_b = encode_indices_b.shape[0]*encode_indices_b.shape[1]*encode_indices_b.shape[2]
        encode_indices_b_flat = encode_indices_b.view(size_encode_indices_b)
        # encode_indices_t shape = [BS * BOTTOM_LATENT_H * BOTTOM_LATENT_W]
        embed_mean_probs    = torch.bincount(encode_indices_b_flat)/size_encode_indices_b
        embed_entropy       = embed_mean_probs * torch.log(embed_mean_probs + 1e-10)
        perplexity_b        = torch.exp(-torch.sum(embed_entropy))
        train_perplexity_b += perplexity_b.item()

        # 8. Save results
        results["train_reconst_loss"].append(reconst_loss.item())
        results["train_loss"].append(loss.item())
        results["train_perplexity_t"].append(perplexity_t.item())
        results["train_perplexity_b"].append(perplexity_b.item())

        # 9. Print progress information
        if i % log_interval == 0:

            print(f'Update #{i: 6d}', end="   ")
            print(f'Reconst loss: {np.mean(results["train_reconst_loss"][-log_interval:]) :0>16.10f}', end="   ")
            print(f'Loss: {np.mean(results["train_loss"][-log_interval:]) :0>16.10f}', end="   ")
            print(f'Perplexity top: {np.mean(results["train_perplexity_t"][-log_interval:]) :0>10.4f}', end="   ")
            print(f'Perplexity bottom: {np.mean(results["train_perplexity_b"][-log_interval:]) :0>10.4f}')

     # 8. Update the learning rate if necessary
    if config["exp_params"]["vqvae_scheduler"] == "RLRonPlateau":
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(loss.item())
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr != old_lr:
            print(f'[INFO] Learning rate was altered from {old_lr} to {new_lr}.')
    elif config["exp_params"]["vqvae_scheduler"] == "CyclicLR":
        scheduler.step()
    elif config["exp_params"]["vqvae_scheduler"] == "CosineAnnealingWarmRestarts":
        scheduler.step(epoch + i / train_data_len)

    # Compute average metrics across all batches
    train_loss         = train_loss / len(dataloader)
    train_reconst_loss = train_reconst_loss / len(dataloader)
    train_perplexity_t = train_perplexity_t / len(dataloader)
    train_perplexity_b = train_perplexity_b / len(dataloader)

    return train_loss, train_reconst_loss , train_perplexity_t, train_perplexity_b

In [ ]:
def val_step_vqvae2(
        model:          torch.nn.Module,
        dataloader:     torch.utils.data.DataLoader,
        epoch:          int,
        log_interval:   int,
        results:        Dict[str, List[float]],
        train_variance: torch.Tensor,
        beta:           float        = 0.25,
        device:         torch.device = device,
    ) -> Tuple[float, float, float]:
    """
    Evaluate a VQ-VAE2 model for a single epoch.

    Turns a VQ-VAE2 'model' to evaluation mode and then performs
    a forward pass on a validation dataset.

    Arguments:
        model:          A VQ-VAE2 model to be evaluated.
        dataloader:     A DataLoader instance for the model to be evaluated on.
        epoch:          The current validation epoch number.
        log_interval:   Interval (in batches) between successive logs.
        results:        Dictionary to append the validation results.
        train_variance: The training dataset variance.
        beta:           Scalar that defines the weight of the commitment term
                        of the loss (variable 'beta' in equation 3 of VQ-VAE paper).
        device:         A target device to compute on (e.g. 'cuda' or 'cpu').

    Returns:
        A tuple of loss and reconstruction loss.
        In the form (loss, reconst_loss).
    """
    # Put the model in evaluation mode
    model.eval()

    # Initialize the validation metrics
    val_loss         = 0
    val_reconst_loss = 0
    #val_perplexity   = 0

    # Turn on inference context manager
    with torch.inference_mode():

        # Loop through the DataLoader batches
        for i, x in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):

            # 1. Send data to target device
            x = x.to(device)

            # 2. Forward pass
            x_pred, latent_loss, enc_ind_t, enc_ind_b = model(x)

            # 3. Calculate the loss
            reconst_loss = torch.mean((x_pred - x)**2) / train_variance
            loss         = reconst_loss + beta * latent_loss

            # 4. Accumulate the metrics
            val_loss         += loss.item()
            val_reconst_loss += reconst_loss.item()
            #val_perplexity   += perplexity.item()

            # 5. Save the partial results
            results["val_reconst_loss"].append(reconst_loss.item())
            results["val_loss"].append(loss.item())
            #results["val_perplexity"].append(perplexity.item())

            # 6. Print progress information
            if i % log_interval == 0:

                print(f'epoch|iter: {epoch+1 :4d} | {i :6d}/{len(dataloader) :6d} ({(i*100)/len(dataloader) :0>5.1f}%)', end="   ")
                print(f'Val reconst loss: {np.mean(results["val_reconst_loss"][-log_interval:]) :0>16.10f})', end="   ")
                print(f'Val loss: {np.mean(results["val_loss"][-log_interval:]) :0>16.10f}')

    # Compute average metrics across all batches
    val_loss         = val_loss / len(dataloader)
    val_reconst_loss = val_reconst_loss / len(dataloader)
    #val_perplexity   = val_perplexity / len(dataloader)

    # Save a sample of 8 images and reconstructions to file at the end of each validation epoch
    sample     = x[:sample_size]
    sample_out = x_pred[:sample_size]
    
    utils.save_image(
        torch.cat([sample, sample_out], 0),
        f'results/reconstruct_{str(epoch + 1).zfill(5)}_{str(i).zfill(5)}.png',
        nrow        = sample_size,
        normalize   = False,
        value_range = (-1, 1),
    )
    
    return val_loss, val_reconst_loss  # , val_perplexity

In [ ]:
def train_vqvae2(
        model:            torch.nn.Module,
        train_dataloader: torch.utils.data.DataLoader,
        val_dataloader:   torch.utils.data.DataLoader,
        optimizer:        torch.optim.Optimizer,
        scheduler:        torch.optim.lr_scheduler.LRScheduler,
        config:           Dict,
        train_variance:   torch.Tensor,
        device:           torch.device,
    ) -> Dict[str, List]:
    """
    Trains and validates a VQ-VAE2 model.

    Passes a target PyTorch models through train_step_vqvae() and val_step_vqvae()
    functions for a number of epochs, training and validating the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Arguments:
        model:            A VQ-VAE2 model to be trained and validated.
        train_dataloader: A DataLoader instance for the model to be trained on.
        test_dataloader:  A DataLoader instance for the model to be tested on.
        optimizer:        A PyTorch optimizer to help minimize the loss function.
        scheduler:        A pytorch learning rate scheduler.
        config:           A Dictionary containing the training configuration.
        train_variance:   The training dataset variance.
        device:           A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A dictionary of training and validation results.
        Each metric has a list of values, one per batch.
        In the form:
            {
            train_reconst_loss: [...],
            train_loss:         [...],
            train_perplexity:   [...],
            val_reconst_loss:   [...],
            val_loss:           [...],
            val_perplexity:     [...],
            }
    """

    log_interval    = config["exp_params"]["log_interval"]
    beta            = config["exp_params"]["beta"] 
    train_variance  = torch.mean(train_variance)
    file_save_model = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'
    n_train_results = log_interval * len(train_dataloader)
    # n_val_results   = log_interval * len(val_dataloader)

    # Create an empty dictionary of results
    results = {
        'train_reconst_loss': [],
        'train_loss':         [],
        'train_perplexity_t': [],
        'train_perplexity_b': [],
        'val_reconst_loss':   [],
        'val_loss':           [],
    }

    for epoch in trange(config["exp_params"]["epochs"]):

        start_time = time.time()
        train_loss, train_reconst_loss, p_t, p_b = train_step_vqvae2(
            model          = model,
            dataloader     = train_dataloader,
            epoch          = epoch,
            log_interval   = log_interval,
            optimizer      = optimizer,
            scheduler      = scheduler,
            results        = results,
            train_variance = train_variance,
            beta           = beta,
            device         = device,
       )

        val_loss, val_reconst_loss = val_step_vqvae2(
            model          = model,
            dataloader     = val_dataloader,
            epoch          = epoch,
            log_interval   = log_interval,
            results        = results,
            train_variance = train_variance,
            beta           = beta,
            device         = device,
        )

        try:
            # Log metrics to W&B
            wandb.log(
                {
                "train_vqvae_reconst_loss":      np.mean(results["train_reconst_loss"][-n_train_results:]),
                "train_vqvae_loss":              np.mean(results["train_loss"][-n_train_results:]),
                "train_vqvae_perplexity_top":    np.mean(results["train_perplexity_t"][-n_train_results:]),
                "train_vqvae_perplexity_bottom": np.mean(results["train_perplexity_b"][-n_train_results:]),
                "val_vqvae_reconst_loss":        np.mean(results["val_reconst_loss"][-n_train_results:]),
                "val_vqvae_loss":                np.mean(results["val_loss"][-n_train_results:]),
                "vqvae_epoch":                   epoch+1,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Save the model, the results, and the hyperparameters

        if config["exp_params"]["save_model"] == True:

            epoch_val_loss =  np.mean(results["val_loss"][-n_train_results:])
            if epoch == 0:
                best_val_loss = epoch_val_loss
                save_model_and_results(
                    model,
                    results,
                    config,
                    file_save_model,
                )

            elif epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                save_model_and_results(
                    model,
                    results,
                    config,
                    file_save_model,
                )

        end_time  = time.time()
        texec_sec = end_time - start_time
        texec_min = int(texec_sec/60)
        texec_sec = int(texec_sec - texec_min * 60)
        
        print(f'Epoch execution time: {texec_min}m {texec_sec}s')

    # Return the filled results at the end of the epochs
    return results

## Instantiate the VQ-VAE 2 model

In [ ]:
vqvae2_model = VQVAE2(
    in_channels      = config["model_params"]["num_channels"],
    channels         = config["model_params"]["hidden_channels"],
    n_res_blocks     = config["model_params"]["num_residual_blocks"],
    n_res_channels   = config["model_params"]["res_hidden_channels"],
    embedding_dim    = config["model_params"]["embedding_dim"],
    embedding_number = config["model_params"]["embedding_number"],
    decay            = config["exp_params"]["decay"],
    eps              = config["exp_params"]["eps"],
    stride_top       = config["model_params"]["stride_top"],
    stride_bottom    = config["model_params"]["stride_bottom"],
    ).to(device)

## Visualize the VQ-VAE 2 model

In [ ]:
file_save_onnx = f'models/{config["exp_params"]["vqvae_save_file"]}.onnx'

vqvae2_model.eval()

visualization_input  = torch.randn(
    config["exp_params"]["train_batch_size"],
    config["model_params"]["num_channels"],
    config["model_params"]["patch_size"],
    config["model_params"]["patch_size"]
).to(device)

input_names  = [ "input_image" ]
output_names = [ "reconstructed_image" ]

torch.onnx.export(
    vqvae2_model,
    visualization_input,
    file_save_onnx,
    verbose=False,
    input_names=input_names,
    output_names=output_names,
    export_params=True,
)

## Train the VQ-VAE 2 model

In [ ]:
# Define the optimizer
optimizer = optim.Adam(
    vqvae2_model.parameters(), 
    lr=config["exp_params"]["learning_rate"]
)

# Define the learning rate scheduler
old_lr = config["exp_params"]["learning_rate"]

if config["exp_params"]["vqvae_scheduler"] == "RLRonPlateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        factor    = 0.75, 
        patience  = 5,
        threshold = 2.5e-3,
    )
elif config["exp_params"]["vqvae_scheduler"] == "CyclicLR":
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr      = config["exp_params"]["learning_rate"], # Initial and minimum learning rate
        max_lr       = 0.01,         # Maximum learning rate
        step_size_up = 5,            # Number of training epochs in the increasing half cycle
        mode         = "triangular", # Can be "triangular" or "triangular2" or "exp_range"
    )
elif config["exp_params"]["vqvae_scheduler"] == "CosineAnnealingWarmRestarts":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0        = 10,    # Number of epochs in the first restart (cycle), which defines the number of 
                            # epochs since LR_max=lr until LR_min=eta_min.
        T_mult     = 1,     # A factor that multiplies T_0 after each restart (cycle), which specifies the
                            # number of epochs in next cycle (default=1).
        eta_min    = 1e-6,  # The minimum learning rate
        last_epoch = -1,    # When last_epoch=-1, sets initial learning rate as the 'lr' value defined on 'optimizer' 
    )
else:
    scheduler = None

In [ ]:
file_saved_model  = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'

if LOAD_VQVAE_MODEL == True:
    _, _ = load_model(vqvae2_model, file_saved_model, config, device)

if SKIP_TRAIN_VQVAE_MODEL == False:
    results = train_vqvae2(
        vqvae2_model,
        train_dataloader,
        val_dataloader,
        optimizer,
        scheduler,
        config,
        train_variance,
        device
    )

### Get the latent representation for all the training set and save it to a LMDB database

## Get and save latent codes for training priors in Stage 2

Parameters:<P>

| name                                           | type | default |
| ---------------------------------------------- | -----| ------- | 
| config["exp_params"]["vqvae_save_file"]        | str  |         |
| config["exp_params"]["pixelsnail_codes_path"]  | str  |         |


In [ ]:
def get_and_save_codes(filename, lmdb_env, loader, model, device):
    index = 0

    with lmdb_env.begin(write=True) as txn:
        pbar = tqdm(loader)

        for img in pbar:
            
            img                 = img.to(device)
            _, _, _, id_t, id_b = model.encode(img)
            id_t                = id_t.detach().cpu().numpy()
            id_b                = id_b.detach().cpu().numpy()

            for file, top, bottom in zip(filename, id_t, id_b):
                row = CodeRow(
                    top      = top,
                    bottom   = bottom,
                    filename = file
                )
                txn.put(str(index).encode('utf-8'), pickle.dumps(row))
                index += 1
                pbar.set_description(f'inserted: {index}')

        txn.put('length'.encode('utf-8'), str(index).encode('utf-8'))

In [ ]:
file_load_model = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'
codes_file      = config["exp_params"]["pixelsnail_codes_path"]

vqvae2_model.to(device)

vqvae2_model.eval()

map_size = 100 * 1024 * 1024 * 1024

env = lmdb.open(codes_file, map_size=map_size)

get_and_save_codes(codes_file, env, train_dataloader, vqvae2_model, device)

## Stage 2: Train the PixelSNAIL models

### LMDB Dataset

In [ ]:
CodeRow = namedtuple('CodeRow', ['top', 'bottom', 'filename'])

class ImageFileDataset(datasets.ImageFolder):

    def __getitem__(self, index):
        sample, target = super().__getitem__(index)
        path, _        = self.samples[index]
        dirs, filename = os.path.split(path)
        _, class_name  = os.path.split(dirs)
        filename       = os.path.join(class_name, filename)

        return sample, target, filename


class LMDBDataset(Dataset):

    def __init__(self, path):
        self.env = lmdb.open(
            path,
            max_readers = 32,
            readonly    = True,
            lock        = False,
            readahead   = False,
            meminit     = False,
        )

        if not self.env:
            raise IOError('Cannot open lmdb dataset', path)

        with self.env.begin(write=False) as txn:
            self.length = int(txn.get('length'.encode('utf-8')).decode('utf-8'))

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        with self.env.begin(write=False) as txn:
            key = str(index).encode('utf-8')

            row = pickle.loads(txn.get(key))

        return torch.from_numpy(row.top), torch.from_numpy(row.bottom), row.filename


### PixelSNAIL model

In [ ]:
"""
Linear layer with weight normalization applied to its weights.
"""
def wn_linear(in_dim, out_dim):
    return nn.utils.weight_norm(nn.Linear(in_dim, out_dim))

def shift_down(input, size=1):
    return F.pad(input, [0, 0, size, 0])[:, :, : input.shape[2], :]


def shift_right(input, size=1):
    return F.pad(input, [size, 0, 0, 0])[:, :, :, : input.shape[3]]

'''
Define a 'mask' and 'start_mask' as follows (when size=5):

mask =
    [
        [
            [0, 0, 0, 0, 0],
            [1, 0, 0, 0, 0],
            [1, 1, 0, 0, 0],
            [1, 1, 1, 0, 0],
            [1, 1, 1, 1, 0]
        ]
    ]

start_mask =
    [
        [0.],
        [1.],
        [1.],
        [1.],
        [1.]
    ]
'''
@lru_cache(maxsize=64)
def causal_mask(size):
    shape         = [size, size]
    mask          = np.triu(np.ones(shape), k=1).astype(np.uint8).T
    start_mask    = np.ones(size).astype(np.float32)
    start_mask[0] = 0

    return (
        torch.from_numpy(mask).unsqueeze(0),
        torch.from_numpy(start_mask).unsqueeze(1),
    )

In [ ]:
class WNConv2d(nn.Module):
    '''
    Convolution layer with weight normalization applied to its weights, 
    and with the selected activation function placed after the convolution.
    '''
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride     = 1,
        padding    = 0,
        bias       = True,
        activation = None,
    ):
        super().__init__()

        # Create a Conv2d layer and apply weight normalization to its parameters
        self.conv = nn.utils.weight_norm(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride  = stride,
                padding = padding,
                bias    = bias,
            )
        )

        self.out_channels = out_channels

        if isinstance(kernel_size, int):
            kernel_size = [kernel_size, kernel_size]

        self.kernel_size = kernel_size

        self.activation = activation

    def forward(self, input):
        out = self.conv(input)

        if self.activation is not None:
            out = self.activation(out)

        return out


In [ ]:
class GatedResBlock(nn.Module):
    '''
    Gated Residual Block.
    '''
    def __init__(
        self,
        in_channels,
        channels,
        kernel_size,
        conv               = 'wnconv2d',
        activation         = nn.ELU,
        dropout            = 0.1,
        auxiliary_channels = 0,
        condition_dim      = 0,
        ):
        super().__init__()

        if conv == 'wnconv2d':
            # Define 'conv_module' as a "partial" function of WNConv2d with 
            # argument 'padding' fixed at 'kernel_size // 2'
            conv_module = partial(WNConv2d, padding=kernel_size // 2)

        elif conv == 'causal_downright':
            # Define 'conv_module' as a "partial" function of CausalConv2d with 
            # argument 'padding' fixed at 'downright'
            conv_module = partial(CausalConv2d, padding='downright')

        elif conv == 'causal':
            # Define 'conv_module' as a "partial" function of CausalConv2d with 
            # argument 'padding' fixed at 'causal'
            conv_module = partial(CausalConv2d, padding='causal')

        self.activation = activation()
        self.conv1      = conv_module(in_channels, channels, kernel_size)

        if auxiliary_channels > 0:
            self.aux_conv = WNConv2d(auxiliary_channels, channels, 1)

        self.dropout = nn.Dropout(dropout)

        self.conv2   = conv_module(channels, in_channels * 2, kernel_size)

        if condition_dim > 0:
             self.condition = WNConv2d(condition_dim, in_channels * 2, 1, bias=False)

        self.gate = nn.GLU(1)

    def forward(self, input, aux_input=None, condition=None):
        # input -> activation -> conv_module -> out 
        out = self.conv1(self.activation(input))

        # aux_input -> activation -> WNConv2d -> aux1
        # out + aux1 -> out
        if aux_input is not None:
            out = out + self.aux_conv(self.activation(aux_input))

        # out -> activation -> Dropout -> activation -> conv_module -> out
        out = self.activation(out)
        out = self.dropout(out)
        out = self.conv2(out)

        # condition  -> WNConv2d -> aux2
        # aux2 + out -> out
        if condition is not None:
            condition = self.condition(condition)
            out      += condition
            # out = out + condition.view(condition.shape[0], 1, 1, condition.shape[1])

        # out -> GLU activation -> out
        # out + input -> out
        out = self.gate(out)
        out += input

        return out


In [ ]:
class CausalConv2d(nn.Module):
    '''
    Causal convolution layer with weight normalization applied to its weights, 
    and with the selected activation function placed after the convolution.
    According to the type of selected padding, input tensor is padded with zeros
    in some sides, and if type of padding is 'causal' some weights of the 
    convolution are set to zero.
    '''
    def __init__(
        self,
        in_channel,
        out_channel,
        kernel_size,
        stride       = 1,
        padding      = 'downright',
        activation   = None,
    ):
        super().__init__()

        if isinstance(kernel_size, int):
            kernel_size = [kernel_size] * 2

        self.kernel_size = kernel_size

        # The padding size specifies the number of values to append 
        # into a given tensor at the [left, right, top, bottom]
        if padding == 'downright':
            pad = [kernel_size[1] - 1, 0, kernel_size[0] - 1, 0]

        elif padding == 'down' or padding == 'causal':
            pad = kernel_size[1] // 2

            pad = [pad, pad, kernel_size[0] - 1, 0]

        self.causal = 0
        if padding == 'causal':
            self.causal = kernel_size[1] // 2

        # Zero padding layer, with the number of zeros to append being given by 'pad'
        self.pad = nn.ZeroPad2d(pad)

        # Conv2d layer with weight normalization and with an activation function
        self.conv = WNConv2d(
            in_channel,
            out_channel,
            kernel_size,
            stride        = stride,
            padding       = 0,
            activation    = activation,
        )

    def forward(self, input):
        # Pad 'input' with zeros at the sides specified by the selected 
        # type of padding ('downright', 'down' or 'causal')
        out = self.pad(input)

        # If the type of padding is 'causal', some of the convolution
        # weights directions ('weight_v') are set to zero
        if self.causal > 0:
            self.conv.conv.weight_v.data[:, :, -1, self.causal :].zero_()

        # Apply convolution to the padded input
        out = self.conv(out)

        return out


In [ ]:
class CausalAttention(nn.Module):
    '''
    PixelSNAIL Causal Attention block.
    Essentially, it implements:
      key       = Linear(WeightNorm(key_input)
      value     = Linear(WeightNorm(key_input)
      query     = Linear(WeightNorm(query_input)
      aux1      = MatrixMult(key,query) / sqrt(dim_head)
      aux2      = Dropout(Softmax(aux1))
      attention = aux2 x value
    '''
    def __init__(self, query_channels, key_channels, channels, n_heads=8, dropout=0.1):
        super().__init__()

        self.query     = wn_linear(query_channels, channels)
        self.key       = wn_linear(key_channels, channels)
        self.value     = wn_linear(key_channels, channels)

        self.dim_head  = channels // n_heads
        self.n_heads   = n_heads

        self.dropout   = nn.Dropout(dropout)

    def forward(self, query, key):
        batch, _, height, width = key.shape

        def reshape(input):
            return input.view(batch, -1, self.n_heads, self.dim_head).transpose(1, 2)

        query_flat = query.view(batch, query.shape[1], -1).transpose(1, 2)
        key_flat   = key.view(batch, key.shape[1], -1).transpose(1, 2)
        query      = reshape(self.query(query_flat))
        key        = reshape(self.key(key_flat)).transpose(2, 3)
        value      = reshape(self.value(key_flat))

        attn       = torch.matmul(query, key) / sqrt(self.dim_head)
        mask, start_mask = causal_mask(height * width)
        mask       = mask.type_as(query)
        start_mask = start_mask.type_as(query)
        # Replaces the values of 'att' with -10000 when 'mask' is zero
        attn       = attn.masked_fill(mask == 0, -1e4)
        attn       = torch.softmax(attn, 3) * start_mask
        attn       = self.dropout(attn)

        out        = attn @ value
        out        = out.transpose(1, 2).reshape(
            batch, height, width, self.dim_head * self.n_heads
        )
        out        = out.permute(0, 3, 1, 2)

        return out


In [ ]:
class PixelBlock(nn.Module):
    """
    PixelSNAIL block.
    """
    def __init__(
        self,
        in_channels,
        channels,
        kernel_size,
        n_res_blocks,
        attention     = True,
        dropout       = 0.1,
        condition_dim = 0,
        ):
        super().__init__()

        resblocks = []
        for i in range(n_res_blocks):

            resblocks.append(
                GatedResBlock(
                    in_channels,
                    channels,
                    kernel_size,
                    conv          = 'causal',
                    dropout       = dropout,
                    condition_dim = condition_dim,
                )
            )

        self.resblocks = nn.ModuleList(resblocks)

        self.attention = attention

        if attention:
            self.key_resblock = GatedResBlock(
                in_channels * 2 + 2, in_channels, 1, dropout=dropout
            )
            self.query_resblock = GatedResBlock(
                in_channels + 2, in_channels, 1, dropout=dropout
            )

            self.causal_attention = CausalAttention(
                in_channels + 2, in_channels * 2 + 2, in_channels // 2, dropout=dropout
            )

            self.out_resblock = GatedResBlock(
                in_channels,
                in_channels,
                1,
                auxiliary_channels = in_channels // 2,
                dropout=dropout,
            )

        else:
            self.out = WNConv2d(in_channels + 2, in_channels, 1)

    def forward(self, input, background, condition=None):
        out = input

        for resblock in self.resblocks:
            out = resblock(out, condition=condition)

        if self.attention:
            key_cat   = torch.cat([input, out, background], 1)
            key       = self.key_resblock(key_cat)
            query_cat = torch.cat([out, background], 1)
            query     = self.query_resblock(query_cat)
            attn_out  = self.causal_attention(query, key)
            out       = self.out_resblock(out, attn_out)

        else:
            bg_cat    = torch.cat([out, background], 1)
            out       = self.out(bg_cat)

        return out

In [ ]:
class CondResNet(nn.Module):
    """
    A Residual network composed of:
    (i) a Conv2d with normalized weights and an activation function in the end.
    (ii) 'n_res_blocks' gated residual blocks.
    """
    def __init__(self, in_channels, channels, kernel_size, n_res_blocks):
        super().__init__()

        blocks = [WNConv2d(in_channels, channels, kernel_size, padding=kernel_size // 2)]

        for i in range(n_res_blocks):
            blocks.append(GatedResBlock(channels, channels, kernel_size))

        self.blocks = nn.Sequential(*blocks)

    def forward(self, input):
        return self.blocks(input)

In [ ]:
class PixelSNAIL(nn.Module):
    """
    PixelSNAIL autoregressive model.
    """
    def __init__(
        self,
        shape,
        n_classes,
        channels,
        kernel_size,
        n_blocks,
        n_res_blocks,
        res_channels,
        attention         = True,
        dropout           = 0.1,
        n_cond_res_blocks = 0,
        cond_res_channels = 0,
        cond_res_kernel   = 3,
        n_out_res_blocks  = 0,
        ):
        super().__init__()

        height, width = shape

        self.n_classes = n_classes

        if kernel_size % 2 == 0:
            kernel = kernel_size + 1

        else:
            kernel = kernel_size

        self.horizontal = CausalConv2d(
            n_classes, channels, [kernel // 2, kernel], padding='down'
        )
        self.vertical = CausalConv2d(
            n_classes, channels, [(kernel + 1) // 2, kernel // 2], padding='downright'
        )

        # Create a grid of 'height' ('width') values, from approximately -0.5 to 0.5, equally spaced.
        # Then, repeat each value 'width' ('height') times to form a matrix.
        # Merge the two matrices and save the result in a buffer of the model.
        coord_x = (torch.arange(height).float() - height / 2) / height
        coord_x = coord_x.view(1, 1, height, 1).expand(1, 1, height, width)
        coord_y = (torch.arange(width).float() - width / 2) / width
        coord_y = coord_y.view(1, 1, 1, width).expand(1, 1, height, width)
        self.register_buffer('background', torch.cat([coord_x, coord_y], 1))

        self.blocks = nn.ModuleList()

        # Create 'n_blocks' PixelSNAIL blocks
        for i in range(n_blocks):

            self.blocks.append(
                PixelBlock(
                    channels,
                    res_channels,
                    kernel_size,
                    n_res_blocks,
                    attention     = attention,
                    dropout       = dropout,
                    condition_dim = cond_res_channels,
                )
            )

        # Residual network with a WNConv2d + 'n_cond_res_blocks' gated residual blocks
        if n_cond_res_blocks > 0:
            self.cond_resnet = CondResNet(
                n_classes, cond_res_channels, cond_res_kernel, n_cond_res_blocks
            )

        out = []

        # 'n_out_res_blocks' gated residual blocks
        for i in range(n_out_res_blocks):
            out.append(GatedResBlock(channels, res_channels, 1))

        # Add ELU activation and WNConv2d layer to the end of 'out' list of modules
        out.extend([nn.ELU(inplace=True), WNConv2d(channels, n_classes, 1)])

        self.out = nn.Sequential(*out)

    def forward(self, input, condition=None, cache=None):
        if cache is None:
            cache = {}
        batch, height, width = input.shape
        input = (
            F.one_hot(input, self.n_classes).permute(0, 3, 1, 2).type_as(self.background)
        )
        horizontal = shift_down(self.horizontal(input))
        vertical   = shift_right(self.vertical(input))
        out        = horizontal + vertical

        background = self.background[:, :, :height, :].expand(batch, 2, height, width)

        if condition is not None:
            if 'condition' in cache:
                condition = cache['condition']
                condition = condition[:, :, :height, :]

            else:
                condition = (
                    F.one_hot(condition, self.n_classes)
                    .permute(0, 3, 1, 2)
                    .type_as(self.background)
                )
                condition = self.cond_resnet(condition)
                condition = F.interpolate(condition, scale_factor=2)
                cache['condition'] = condition.detach().clone()
                condition = condition[:, :, :height, :]

        for block in self.blocks:
            out = block(out, background, condition=condition)

        out = self.out(out)

        return out, cache

### Train the PixelSNAIL models

Parameters: <P>

| name                                                 | type  | default |
| ---------------------------------------------------- | ----- | ------- |
| config["exp_params"]["pixelsnail_train_batch_size"]  | int   | 32      |
| config["exp_params"]["pixelsnail_epochs"]            | int   | 400     |
| config["exp_params"]["pixelsnail_hier_level"]        | str   | 'top'   |
| config["exp_params"]["pixelsnail_lr"]                | float | 3e-4    |
| config["model_params"]["pixelsnail_channels"]        | int   | 128     |
| config["model_params"]["pixelsnail_res_blocks"]      | int   | 4       |
| config["model_params"]["pixelsnail_res_channels"]    | int   | 128     |
| config["model_params"]["pixelsnail_out_res_blocks"]  | int   | 0       |
| config["model_params"]["pixelsnail_cond_res_blocks"] | int   | 3       |
| config["exp_params"]["pixelsnail_dropout"]           | float | 0.1     |
| config["model_params"]["pixelsnail_amp_opt_level"]   | str   | 'O0'    |
| config["exp_params"]["pixelsnail_scheduler"]         | str   |         |
| config["exp_params"]["pixelsnail_save_file"]         | str   |         |
| config["exp_params"]["pixelsnail_codes_path"]        | str   |         |


In [ ]:
class PixelTransform:
    def __init__(self):
        pass

    def __call__(self, input):
        ar = np.array(input)
        return torch.from_numpy(ar).long()

In [ ]:
def train_pixelsnail(loader, model, optimizer, scheduler, device=device):

    log_interval     = config["exp_params"]["log_interval"]
    epochs           = config["exp_params"]["pixelsnail_epochs"]

    if config["exp_params"]["pixelsnail_hier_level"] == 'top':
        file_save_model  = f'models/{config["exp_params"]["pixelsnail_save_file"]}_top.pth'
    elif config["exp_params"]["pixelsnail_hier_level"] == 'bottom':
        file_save_model  = f'models/{config["exp_params"]["pixelsnail_save_file"]}_bottom.pth'

    pixelsnail_results = {
        'loss': [],
        'acc':  [],
    }

    model.train()
    loss_fn = nn.CrossEntropyLoss()

    for epoch in trange(epochs):

        train_loss = 0.0
        train_acc  = 0.0

        for i, (top, bottom, label) in enumerate(tqdm(loader, desc=f'Epoch {epoch+1}')):

            start_time = time.time()

            # 1. Reset the loss gradients
            model.zero_grad()

            # 2. Put the 'top' indices in the 'device'
            top = top.to(device)

            if config["exp_params"]["pixelsnail_hier_level"] == 'top':
                # 3a. Pass the 'top' indices through the top PixelSNAIL
                target = top
                out, _ = model(top)

            elif config["exp_params"]["pixelsnail_hier_level"] == 'bottom':
                # 3b1. Put the 'bottom' indices in the 'device'
                bottom = bottom.to(device)
                target = bottom
                # 3b2. Pass the 'bottom' indices through the bottom PixelSNAIL
                out, _ = model(bottom, condition=top)

            # 4. Calculate the cross-entropy loss between input indices and predicted indices
            loss = loss_fn(out, target)

            # 5. Calculate the gradient of the loss relative to the model parameters
            loss.backward()

            # 6. Update model parameters using the calculated gradients
            optimizer.step()

            # 7. Update the learning rate if necessary
            if config["exp_params"]["pixelsnail_scheduler"] == "RLRonPlateau":
                old_lr = optimizer.param_groups[0]['lr']
                scheduler.step(loss)
                new_lr = optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'[INFO] Learning rate was altered from {old_lr} to {new_lr}.')
            elif config["exp_params"]["pixelsnail_scheduler"] == "CyclicLR":
                scheduler.step()
            elif config["exp_params"]["pixelsnail_scheduler"] == "CosineAnnealingWarmRestarts":
                scheduler.step(epoch + i / train_data_len)

            # 8. Calculate the accuracy
            _, pred  = out.max(1)
            correct  = (pred == target).float()
            acc      = correct.sum() / target.numel()

            # 9. Accumulate and save the loss and accuracy
            train_loss += loss.item()
            train_acc  += acc
            pixelsnail_results["loss"].append(loss.item())
            pixelsnail_results["acc"].append(acc)

            # 10. Display progress information
            if (i + 1) % log_interval == 0:

                end_time = time.time()

                print(f'Iteration: [{i+1 :6d}/{len(loader) :6d} ({((i+1)*100) / len(loader) :0>5.1f}%)]', end="    ")
                print(f'Loss: {np.asarray(pixelsnail_results["loss"])[-log_interval:].mean(0) :0>16.10f}', end="    ")
                print(f'Time: {end_time - start_time :.5f}')

            # 11. Log the metrics to W&B
            wandb.log(
                {
                "pixelsnail_train_loss": loss.item(),
                "pixelsnail_train_acc":  acc,
                "pixelsnail_epoch":      epoch+1,
                "pixelsnail_batch":      i+1,
                }
            )

        # Compute the average loss and average accuracy across all batches
        train_loss  = train_loss / len(loader)
        train_acc   = train_acc / len(loader)

        # Save the model and the hyperparameters to file
        if epoch == 0:
            best_loss = train_loss
            print(f"[INFO] Saving best model to file {file_save_model}")
            torch.save(
                {
                    'model': model.state_dict(),
                    'config': config,
                },
                file_save_model,
            )

        elif train_loss < best_loss:
            best_loss = train_loss
            print(f"[INFO] Saving best model to file {file_save_model}")
            torch.save(
                {
                    'model': model.state_dict(),
                    'config': config,
                },
                file_save_model,
            )

    return pixelsnail_results

In [ ]:
if SKIP_TRAIN_PIXELSNAIL_MODEL == False:

    dataset = LMDBDataset(config["exp_params"]["pixelsnail_codes_path"])

    loader  = DataLoader(
        dataset,
        batch_size  = config["exp_params"]["pixelsnail_train_batch_size"],
        shuffle     = True,
        num_workers = 4,
        drop_last   = True
    )

    ckpt = {}

    if config["exp_params"]["pixelsnail_hier_level"] == 'top':

        file_load_model  = f'models/{config["exp_params"]["pixelsnail_save_file"]}_top.pth'

        model = PixelSNAIL(
            shape            = config["model_params"]["top_latent_shape"],
            n_classes        = config["model_params"]["embedding_number"],
            channels         = config["model_params"]["pixelsnail_channels"],
            kernel_size      = 5,
            n_blocks         = 4,
            n_res_blocks     = config["model_params"]["pixelsnail_res_blocks"],
            res_channels     = config["model_params"]["pixelsnail_res_channels"],
            attention        = True,
            dropout          = config["exp_params"]["pixelsnail_dropout"],
            n_out_res_blocks = config["model_params"]["pixelsnail_out_res_blocks"],
        )

    elif config["exp_params"]["pixelsnail_hier_level"] == 'bottom':

        file_load_model  = f'models/{config["exp_params"]["pixelsnail_save_file"]}_bottom.pth'
        
        model = PixelSNAIL(
            shape             = config["model_params"]["bottom_latent_shape"],
            n_classes         = config["model_params"]["embedding_number"],
            channels          = config["model_params"]["pixelsnail_channels"],
            kernel_size       = 5,
            n_blocks          = 4,
            n_res_blocks      = config["model_params"]["pixelsnail_res_blocks"],
            res_channels      = config["model_params"]["pixelsnail_res_channels"],
            attention         = False,
            dropout           = config["exp_params"]["pixelsnail_dropout"],
            n_cond_res_blocks = config["model_params"]["pixelsnail_cond_res_blocks"],
            cond_res_channels = config["model_params"]["pixelsnail_res_channels"],
        )

    if LOAD_PIXELSNAIL_MODEL == True:
        ckpt   = torch.load(file_load_model)

    if 'model' in ckpt:
        model.load_state_dict(ckpt['model'])
        config = ckpt['config']

    model     = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=config["exp_params"]["pixelsnail_lr"])

    if amp is not None:
        model, optimizer = amp.initialize(
            model, 
            optimizer, 
            opt_level=config["model_params"]["pixelsnail_amp_opt_level"]
        )

    model = model.to(device)

    # Define the learning rate scheduler

    old_lr = config["exp_params"]["pixelsnail_lr"]

    if config["exp_params"]["pixelsnail_scheduler"] == "RLRonPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode      = 'min',
            factor    = 0.9, 
            patience  = 5,
            threshold = 2.5e-3,
        )
    elif config["exp_params"]["pixelsnail_scheduler"] == "CyclicLR":
        scheduler = torch.optim.lr_scheduler.CyclicLR(
            optimizer,
            base_lr      = config["exp_params"]["pixelsnail_lr"], # Initial and minimum learning rate
            max_lr       = 0.01,         # Maximum learning rate
            step_size_up = 5,            # Number of training iterations/epochs in the increasing half cycle
            mode         = "triangular", # Can be "triangular" or "triangular2" or "exp_range"
        )
    elif config["exp_params"]["pixelsnail_scheduler"] == "CosineAnnealingWarmRestarts":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, 
            T_0        = 10,    # Number of iterations/epochs in the first restart (cycle), which defines the number of 
                                # iterations/epochs since LR_max=lr until LR_min=eta_min.
            T_mult     = 1,     # A factor that multiplies T_0 after each restart (cycle), which specifies the
                                # number of iterations/epochs in next cycle (default=1).
            eta_min    = 1e-6,  # The minimum learning rate
            last_epoch = -1,    # When last_epoch=-1, sets initial learning rate as the 'lr' value defined on 'optimizer' 
        )
    else:
        scheduler = None

    pixelsnail_results = train_pixelsnail(
        loader    = loader,
        model     = model,
        optimizer = optimizer,
        scheduler = scheduler,
        device    = device,
    )


## Generate samples with the trained models

In [ ]:
@torch.no_grad()
def sample_model(model, num_samples, latent_shape, temperature, condition=None, device=device):
    '''
    Draw 'num_samples' samples from the trained PixelSNAIL 'model'.
    '''
    row   = torch.zeros(num_samples, *latent_shape, dtype=torch.int64).to(device)
    cache = {}

    for i in trange(latent_shape[0], desc='Latent H'):

        for j in trange(latent_shape[1], desc='Latent W'):

            out, cache   = model(row[:, : i + 1, :], condition=condition, cache=cache)
            probs        = torch.softmax(out[:, :, i, j] / temperature, 1)
            sample       = torch.multinomial(probs, 1).squeeze(-1)
            row[:, i, j] = sample

    return row


In [ ]:
def load_model(model_name, model_file, config, device=device):
    '''
    Load a saved model (VQ-VAE or PixelSNAIL top/bottom) from file.
    '''
    loaded_obj = torch.load(model_file)

    if 'config' in loaded_obj:
        config_loaded = loaded_obj['config']

    if model_name == 'vqvae':
        model = VQVAE2(
            in_channels      = config["model_params"]["num_channels"],
            channels         = config["model_params"]["hidden_channels"],
            n_res_blocks     = config["model_params"]["num_residual_blocks"],
            n_res_channels   = config["model_params"]["res_hidden_channels"],
            embedding_dim    = config["model_params"]["embedding_dim"],
            embedding_number = config["model_params"]["embedding_number"],
            decay            = config["exp_params"]["decay"],
            eps              = config["exp_params"]["eps"],
            stride_top       = config["model_params"]["stride_top"],
            stride_bottom    = config["model_params"]["stride_bottom"],
        )

    elif model_name == 'top':
        model = PixelSNAIL(
            shape            = config["model_params"]["top_latent_shape"],
            n_classes        = config["model_params"]["embedding_number"],
            channels         = config["model_params"]["pixelsnail_channels"],
            kernel_size      = 5,
            n_blocks         = 4,
            n_res_blocks     = config["model_params"]["pixelsnail_res_blocks"],
            res_channels     = config["model_params"]["pixelsnail_res_channels"],
            attention        = True,
            dropout          = config["exp_params"]["pixelsnail_dropout"],
            n_out_res_blocks = config["model_params"]["pixelsnail_out_res_blocks"],
        )

    elif model_name == 'bottom':
        model = PixelSNAIL(
            shape             = config["model_params"]["bottom_latent_shape"],
            n_classes         = config["model_params"]["embedding_number"],
            channels          = config["model_params"]["pixelsnail_channels"],
            kernel_size       = 5,
            n_blocks          = 4,
            n_res_blocks      = config["model_params"]["pixelsnail_res_blocks"],
            res_channels      = config["model_params"]["pixelsnail_res_channels"],
            attention         = False,
            dropout           = config["exp_params"]["pixelsnail_dropout"],
            n_cond_res_blocks = config["model_params"]["pixelsnail_cond_res_blocks"],
            cond_res_channels = config["model_params"]["pixelsnail_res_channels"],
        )
       
    if 'model' in loaded_obj:
        loaded_obj = loaded_obj['model']

    model.load_state_dict(loaded_obj)
    model = model.to(device)
    model.eval()

    return model

In [ ]:
'''
Load the VQ-VAE, the PixelSNAIL top and the PixelSNAIL bottom from files.
'''
if GENERATE_NEW_IMAGES == True:

    vqvae_model_file             = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'
    pixelsnail_top_model_file    = f'models/{config["exp_params"]["pixelsnail_save_file"]}_top.pth'
    pixelsnail_bottom_model_file = f'models/{config["exp_params"]["pixelsnail_save_file"]}_bottom.pth'
    
    # (model_name, model_file, config, device=device
    model_vqvae  = load_model(
        'vqvae',
        vqvae_model_file,
        config,
        device = device
    )
    
    model_top    = load_model(
        'top',
        pixelsnail_top_model_file,
        config       = config,
        device       = device
    )
    
    model_bottom = load_model(
        'bottom',
        pixelsnail_bottom_model_file,
        config       = config,
        device       = device
    )

In [ ]:
'''
Get samples from the PixelSNAIL top and then get samples from PixelSNAIL bottom
conditioned by the samples we got from PixelSNAIL top.
Generate new images by passing the samples through the VQ-VAE2 decoder.
'''
if GENERATE_NEW_IMAGES == True:

    NUM_RUNS = 8
    
    for run in tqdm(range(NUM_RUNS), desc="Run: "):
        top_sample    = sample_model(
            model_top,
            num_samples  = 64,
            latent_shape = config["model_params"]["top_latent_shape"],
            temperature  = 1.0,
            device       = device,
        )
        
        bottom_sample = sample_model(
            model_bottom,
            num_samples  = 64,
            latent_shape = config["model_params"]["bottom_latent_shape"],
            temperature  = 1.0,
            condition    = top_sample,
            device       = device,
        )
        
        decoded_sample = model_vqvae.decode_code(top_sample, bottom_sample)
        decoded_sample = decoded_sample.clamp(-1, 1)
    
        SUFFIX   = f'{run}'
        SUFFIX   = SUFFIX.zfill(3)
        SUFFIX   ="_generated_" + SUFFIX + ".png"
        file_png = f'results/{config["exp_params"]["vqvae_save_file"]}{SUFFIX}'
    
        save_image(decoded_sample, file_png, normalize=False, value_range=(-1, 1))

In [ ]:
# Mark the W&B run as finished
wandb.finish()